# Results for model: Qwen/Qwen2.5-Coder-32B-Instruct

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

# Load data
df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Feature engineering - calculate top quantile of feature_00 relative to responder_6
def calculate_top_quantile(group):
    # Calculate quantile threshold for this group
    quantile_threshold = np.quantile(group['feature_00'], 0.85)
    return quantile_threshold

# Group by date_id and calculate quantile threshold
quantile_thresholds = df.groupby('date_id').apply(
    lambda group: calculate_top_quantile(group)
).to_series()

# Create mapping from date_id to quantile threshold
date_to_quantile = dict(zip(quantile_thresholds[0].to_list(), quantile_thresholds[1].to_list()))

# Add quantile column to dataframe
df = df.with_columns([
    pl.col('date_id').map_elements(lambda x: date_to_quantile[x], return_dtype=pl.Float64).alias('top_quantile')
])

# Prepare features and target
feature_cols = [col for col in df.columns if col.startswith('feature_')]
X = df.select(feature_cols).to_numpy()
y = df.select('responder_6').to_numpy().ravel()

# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X, y)

# Predict and evaluate
y_pred = model.predict(X)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE: {rmse}")